# Average feature importance across all urban forms for every BSU

In [ ]:
import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Load models

In [ ]:
fi = {}
lc = {}
perf = []

for reduction in ["fa", "pca"]:
    fi[reduction] = {}
    lc[reduction] = {}
    for model_type in ["lr", "rf"]:
        fi[reduction][model_type] = {}
        lc[reduction][model_type] = {}
        for cluster in [1, 3, 4, 5, 6, 7, 8]:
            with open(
                f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib",
                "rb",
            ) as f:
                model = joblib.load(f)
                if model_type == "rf":
                    fi[reduction][model_type][cluster] = model.feature_importances_
                else:
                    lc[reduction][model_type][cluster] = model.local_coef_
                perf.append(
                    pd.Series(
                        {
                            "reduction": reduction,
                            "model": model_type,
                            "cluster": cluster,
                            "accuracy": model.score_,
                            "balanced_accuracy": model.balanced_accuracy_,
                            "precision": model.precision_,
                            "recall": model.recall_,
                            "f1_macro": model.f1_macro_,
                            "f1_micro_": model.f1_micro_,
                            "f1_weighted": model.f1_weighted_,
                        }
                    )
                )

## Load Data

In [ ]:
census = gpd.read_parquet(
    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
)

selection = [
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Zaměstnaní - Specialisté",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý",
    "Počet osob v bytech celkem  s právním důvodem užívání: družstevní",
    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba",
    "Počet obyvatel na dům",
    "Obyvatelstvo - věk: 7 - 14  - celkem",
    "Obyvatelstvo - věk: 15 - 24  - celkem",
    "Obyvatelstvo - věk: 45 - 54  - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby v domácnosti, děti předškolního věku, ostatní závislé osoby - celkem",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem",
    "Obyvatelstvo - s dlouhodobým pobytem - celkem",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem",
    "Obyvatelstvo - rodinný stav: rozvedení - celkem",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem",
    "geometry",
]
fas = census[selection]

## Merge data with clusters

In [ ]:
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = fas.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(["geometry", "kod_nadzsj_d", "final_without_noise"])

mapped = data["final_without_noise"].map(cluster_mapping[3])

In [ ]:
lc_mean = pd.concat(lc["fa"]["lr"]).groupby(level=1).mean()
lc_std = pd.concat(lc["fa"]["lr"]).groupby(level=1).std()

In [ ]:
lc_mean_mean = lc_mean.mean(axis=0)
lc_mean_mean.reindex(lc_mean_mean.abs().sort_values(ascending=False).index).head(5)

In [ ]:
dfs = []

for i in list(lc["fa"]["lr"].keys()):
    imp = lc["fa"]["lr"][i].abs().mean(axis=0)
    imp = pd.DataFrame(imp, columns=[str(i)])
    dfs.append(imp)

abs_mean = pd.concat(dfs, axis=1)
## nejvyšší vliv mají obecně tyhle proměnný
abs_mean.mean(axis=1).sort_values(ascending=False).head(10)

In [ ]:
dfs = []

for i in list(lc["fa"]["lr"].keys()):
    imp = lc["fa"]["lr"][i].mean(axis=0)
    imp = pd.DataFrame(imp, columns=[str(i)])
    dfs.append(imp)

mean = pd.concat(dfs, axis=1)
mean.std(axis=1).sort_values(ascending=False).head(10)

In [ ]:
lc_mean = lc_mean.set_geometry(data.geometry)
lc_std = lc_std.set_geometry(data.geometry)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))


lc_mean.plot(
    ax=axes[0],
    column="Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    legend=True,
    cmap="coolwarm_r",
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
)
axes[0].set_axis_off()
axes[0].set_title("mean")


lc_std.plot(
    ax=axes[1],
    column="Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    legend=True,
    cmap="YlOrRd",
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
)
axes[1].set_axis_off()
axes[1].set_title("std")

plt.tight_layout()
# fig.savefig("side_by_side.png", dpi=300, bbox_inches="tight")
plt.show()

## Calculate the avg and std for feature importances

In [ ]:
fi_mean = pd.concat(fi["fa"]["rf"]).groupby(level=1).mean()
fi_std = pd.concat(fi["fa"]["rf"]).groupby(level=1).std()

In [ ]:
fi_mean_std = fi_std.mean(axis=0)
fi_mean_std.sort_values(ascending=False).head(10)

Get the variables with overall highest importance

In [ ]:
dfs = []

for i in list(fi["fa"]["rf"].keys()):
    imp = fi["fa"]["rf"][i].abs().mean(axis=0)
    imp = pd.DataFrame(imp, columns=[str(i)])
    dfs.append(imp)

abs_mean = pd.concat(dfs, axis=1)
pd.DataFrame(abs_mean.mean(axis=1).sort_values(ascending=False).head(15).round(4))

Get the variables with overall highest std deviation of the importance

In [ ]:
dfs = []

for i in list(fi["fa"]["rf"].keys()):
    imp = fi["fa"]["rf"][i].mean(axis=0)
    imp = pd.DataFrame(imp, columns=[str(i)])
    dfs.append(imp)

mean = pd.concat(dfs, axis=1)
pd.DataFrame(mean.std(axis=1).sort_values(ascending=False).head(10))

### Assign geometry

In [ ]:
fi_mean = fi_mean.set_geometry(data.geometry)
fi_std = fi_std.set_geometry(data.geometry)

In [ ]:
ax = fi_mean.plot(
    figsize=(15, 10),
    column="Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý",
    legend=True,
    cmap="RdYlGn_r",
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
    #  scheme="natural_breaks"
)
ax.set_axis_off()

fig = ax.get_figure()
# fig.savefig("rentals_imp.png", dpi=300)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))


fi_mean.plot(
    ax=axes[0],
    column="Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    legend=True,
    cmap="RdYlGn_r",
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
)
axes[0].set_axis_off()
axes[0].set_title("mean")


fi_std.plot(
    ax=axes[1],
    column="Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    legend=True,
    cmap="YlOrRd",
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
)
axes[1].set_axis_off()
axes[1].set_title("std")

plt.tight_layout()
# fig.savefig("side_by_side.png", dpi=300, bbox_inches="tight")
plt.show()